# ISS Group 24 Modeling — OWLv2 Few-Shot Single-Object Localizer

**Architecture**: `google/owlv2-base-patch16-ensemble` (frozen) + Multi-View Aggregator (~3M params) + Existence Head (~5K params), with optional LoRA on the last 4 ViT blocks at Stage 2.3.

**Pipeline**:
1. **Phase 0** — zero-shot OWLv2 baseline (no training).
2. **Stage 1.1** — train aggregator + existence head only; backbone & detection heads frozen.
3. **Stage 1.2** — also fine-tune OWLv2 box / class heads.
4. **Stage 2.3** — also LoRA-fine-tune the last 4 vision transformer blocks.

Each stage runs `epochs × K=5 folds`. Every (epoch, fold) writes a checkpoint and an analytics JSON.

**Before running on Colab:**
1. Open the [shared Drive folder](https://drive.google.com/drive/folders/19el_ISdRf5WoXarBsUOXjxFY9A4cPGYJ?usp=sharing) and click **"Add shortcut to Drive"**.
2. *Runtime → Change runtime type → T4 GPU*.
3. Run cells **in order** top-to-bottom. Each stage cell is idempotent — re-running a completed stage skips it via checkpoint detection.

In [1]:
import os
import sys
import subprocess
from pathlib import Path

In [2]:
SHARED_FOLDER_LINK = "https://drive.google.com/drive/folders/19el_ISdRf5WoXarBsUOXjxFY9A4cPGYJ?usp=sharing"
REPO_URL = "https://github.com/pewpewnor/iss_group_24.git"
REPO_LOCAL_PATH = "/content/iss_group_24_src"
USE_GOOGLE_COLAB = False

In [3]:
if USE_GOOGLE_COLAB:
    from google.colab import drive


    def mount_drive() -> str:
        """Mount Google Drive and return the iss_group_24 project root path."""
        drive.mount("/content/drive")
        for candidate in [
            "/content/drive/Shareddrives/iss_group_24",
            "/content/drive/MyDrive/iss_group_24",
        ]:
            if Path(candidate).exists():
                print(f"Drive mounted. Project root: {candidate}")
                return candidate
        raise RuntimeError(
            "Shared folder 'iss_group_24' not found on this Drive.\n"
            f"Open the link and add a shortcut to your Drive: {SHARED_FOLDER_LINK}"
        )

    DRIVE_ROOT = mount_drive()
else:
    DRIVE_ROOT = "."

In [4]:
def setup_repo() -> None:
    """Clone the repo at depth=1, or reset to origin/main if already cloned."""
    if Path(REPO_LOCAL_PATH).exists():
        subprocess.run(["git", "-C", REPO_LOCAL_PATH, "fetch", "origin"], check=True)
        subprocess.run(
            ["git", "-C", REPO_LOCAL_PATH, "reset", "--hard", "origin/main"],
            check=True,
        )
        print(f"Repo updated to origin/main: {REPO_LOCAL_PATH}")
    else:
        subprocess.run(
            ["git", "clone", "--depth=1", REPO_URL, REPO_LOCAL_PATH],
            check=True,
        )
        print(f"Repo cloned: {REPO_LOCAL_PATH}")
    sys.path.insert(0, REPO_LOCAL_PATH)


def install_deps() -> None:
    """Install packages not pre-installed on Colab. Torch / torchvision come pre-installed."""
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "pillow>=10.0", "scipy",
            "transformers>=4.40", "accelerate>=0.30",
            "peft>=0.10", "huggingface_hub>=0.23",
            "matplotlib>=3.7",
        ],
        check=True,
    )
    print("Dependencies ready")

In [5]:
MANIFEST  = f"{DRIVE_ROOT}/dataset/aggregated/manifest.json"
DATA_ROOT = f"{DRIVE_ROOT}/dataset/aggregated"
OUT_ROOT  = f"{DRIVE_ROOT}/checkpoints"
ANALYSIS_ROOT = f"{DRIVE_ROOT}/analysis"

# Set CWD to the project root so relative paths in train.py land here.
os.chdir(DRIVE_ROOT)
print(f"Working directory: {os.getcwd()}")

Working directory: /mnt/Onetera/iss_group_24


In [6]:
if USE_GOOGLE_COLAB:
    setup_repo()
    install_deps()

---
## Step 0 — Build Dataset Manifest (80/20 train/test + phase0)

Stages HOTS + InsDet into `dataset/aggregated/{train,test}` and vizwiz_novel into `dataset/aggregated/phase0`.  **Run once.** Idempotent — re-runs only when the staging directory is missing.

In [7]:
import aggregator

train_dir = Path(DATA_ROOT) / 'train'
test_dir  = Path(DATA_ROOT) / 'test'
phase0_dir = Path(DATA_ROOT) / 'phase0'
staged_ok = (
    Path(MANIFEST).exists()
    and train_dir.is_dir() and any(train_dir.iterdir())
    and test_dir.is_dir() and any(test_dir.iterdir())
    and phase0_dir.is_dir() and any(phase0_dir.iterdir())
)
if staged_ok:
    print(f'Dataset already staged at {DATA_ROOT} — skipping aggregator.')
else:
    print('Running aggregator…')
    aggregator.main()

Dataset already staged at ./dataset/aggregated — skipping aggregator.


---
## Imports & shared kwargs

Every hyperparameter is here. Each per-stage cell pulls from `SHARED_KWARGS`; override with cell-local kwargs.

Defaults assume a 6 GB laptop GPU (RTX 2060): `img_size=768, batch_size=1, grad_accum_steps=8`. Bump these on Colab T4 (`img_size=960, batch_size=4, grad_accum_steps=2`).

In [8]:
import torch
from modeling.train import (
    train_phase0,
    train_stage_1_1,
    train_stage_1_2,
    train_stage_2_3,
    evaluate_phase0,
    evaluate_run,
)
from modeling.plot import plot_all_from_jsons

/mnt/Onetera/iss_group_24/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# VRAM budget (AMP fp16, train+backward, measured on RTX 2060 6 GB):
#   Stage 1.1/1.2  img=960 B=4 → 2.72 GB  ✓
#   Stage 2.3      img=960 B=2 → 5.60 GB  (risky, skip)
#   Stage 2.3      img=768 B=2 → 3.89 GB  ✓  ← safe
#
# OWLv2 snaps input sizes to multiples of patch_size=16 internally, so
# img=780 and img=768 are identical (both → 2304 patches).  Use 768
# explicitly.  Switching resolution between stages is safe: the model
# uses interpolate_pos_encoding=True and attention-based (set-size-
# agnostic) pooling in the aggregator, so no weights are shape-tied
# to the patch count.
_VRAM = torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else 0
_HAS_BIG_GPU = _VRAM >= 16e9

if _HAS_BIG_GPU:
    _DEFAULT_IMG_SIZE, _DEFAULT_BATCH, _DEFAULT_ACCUM = 960, 4, 2
    _S23_IMG_SIZE,     _S23_BATCH,     _S23_ACCUM     = 960, 4, 2
elif _VRAM >= 5.5e9:
    # Medium-tier (RTX 2060 6 GB).  We tested img=960 (60×60 patches,
    # ~2.7 GB at B=4) hoping the extra patches would help InsDet's tiny
    # objects, but a 1-instance A/B showed pos/neg score gap going
    # *negative* on InsDet at 960 vs slightly positive at 768.  Cropping
    # supports tightly (A1) was also tested and made things worse.  Both
    # interventions reverted; defaults below are the original conservative
    # config that gave HOTS map_50 = 0.42 in Phase 0.
    _DEFAULT_IMG_SIZE, _DEFAULT_BATCH, _DEFAULT_ACCUM = 768, 4, 4
    _S23_IMG_SIZE,     _S23_BATCH,     _S23_ACCUM     = 768, 2, 4
else:
    _DEFAULT_IMG_SIZE, _DEFAULT_BATCH, _DEFAULT_ACCUM = 768, 2, 8
    _S23_IMG_SIZE,     _S23_BATCH,     _S23_ACCUM     = 768, 2, 8

_tier = 'large' if _HAS_BIG_GPU else ('medium' if _VRAM >= 5.5e9 else 'small')
print(f'GPU tier: {_tier} ({_VRAM/1e9:.1f} GB)')
print(f'  stages 1.1/1.2 : img={_DEFAULT_IMG_SIZE}  batch={_DEFAULT_BATCH}  accum={_DEFAULT_ACCUM}')
print(f'  stage  2.3     : img={_S23_IMG_SIZE}  batch={_S23_BATCH}  accum={_S23_ACCUM}')

GPU tier: medium (6.0 GB)
  stages 1.1/1.2 : img=768  batch=4  accum=4
  stage  2.3     : img=768  batch=2  accum=4


In [10]:
SHARED_KWARGS = dict(
    manifest        = MANIFEST,
    data_root       = DATA_ROOT,
    out_root        = OUT_ROOT,
    analysis_root   = ANALYSIS_ROOT,

    # Hardware
    img_size         = _DEFAULT_IMG_SIZE,
    batch_size       = _DEFAULT_BATCH,
    grad_accum_steps = _DEFAULT_ACCUM,
    num_workers      = 2,
    use_amp          = True,

    # Folds (every stage runs `epochs × K folds`; user contract).
    folds            = 5,
    fold_seed        = 42,
    n_support        = 4,
    neg_prob         = 0.5,

    # Per-fold episode budgets (stage epochs themselves are passed at
    # the train call site below — see Stage 1.1 / 1.2 / 2.3 cells).
    episodes_per_fold_s1 = 200,
    episodes_per_fold_s2 = 250,
    episodes_per_fold_s3 = 250,
    val_episodes     = 100,

    # LRs (per stage).  Stage 1.1's existence head LR is bumped *much*
    # higher than the aggregator's because it is the only learner driving
    # the focal loss in 1.1; weight decay on the existence head is also
    # disabled (in `_optim.py`) to avoid pulling its bias toward zero.
    lr_aggregator_s1 = 5e-4,
    lr_existence_s1  = 2e-3,
    lr_aggregator_s2 = 1e-4,
    lr_existence_s2  = 1e-4,
    lr_box_s2        = 5e-5,
    lr_class_s2      = 5e-5,
    lr_aggregator_s3 = 5e-5,
    lr_existence_s3  = 1e-4,
    lr_box_s3        = 2e-5,
    lr_class_s3      = 2e-5,
    lr_lora_s3       = 1e-4,
    weight_decay     = 1e-4,
    grad_clip        = 1.0,
    warmup_frac      = 0.05,

    # LoRA (Stage 2.3).
    lora_r           = 8,
    lora_alpha       = 16,
    lora_dropout     = 0.1,
    lora_layers      = 4,

    # Loss — base detection terms.
    focal_alpha      = 0.25,
    focal_gamma      = 2.0,
    lambda_l1        = 5.0,
    lambda_giou      = 2.0,
    anti_collapse_weight   = 0.1,
    box_size_threshold     = 0.6,
    existence_kl_threshold = 0.85,

    # Loss — separation / discriminative terms.  These directly attack the
    # observed Stage 1.1 failure modes:
    #   margin_*       — per-sample hinge: pushes pos logits above
    #                    +margin/2 and neg logits below -margin/2.
    #                    Works at any batch size (in particular B=1
    #                    where a batch is all-pos or all-neg).
    #   contrastive_*  — NT-Xent on prototypes; pushes the aggregator to
    #                    produce *instance*-discriminative embeddings.
    #                    Silently no-ops when batch_size < 2 (laptop
    #                    regime) — relies on the margin term there.
    margin_weight    = 0.5,
    margin_value     = 1.0,
    contrastive_weight = 0.1,
    contrastive_temp   = 0.1,

    # Early stopping.
    early_stop_patience_s1 = 4,
    early_stop_patience_s2 = 5,
    early_stop_patience_s3 = 4,

    tile_cfg = {'mode': 'off'},

    # I/O.
    # keep_last_n=0 ⇒ keep every per-(epoch, fold) rolling ckpt.  This
    # is the right default for Colab / Drive where the runtime can be
    # killed mid-stage and any fold's checkpoint may be the resume
    # target.  Set to a positive integer if local disk is tight.
    keep_last_n      = 0,
    seed             = 42,
    device           = None,    # auto cuda / cpu
)

In [11]:
EVAL_KWARGS = dict(
    manifest    = MANIFEST,
    data_root   = DATA_ROOT,
    out_root    = OUT_ROOT,
    img_size    = _DEFAULT_IMG_SIZE,
    batch_size  = _DEFAULT_BATCH,
    num_workers = 2,
    n_support   = 4,
    neg_prob    = 0.5,
    test_episodes = 400,
    seed        = 42,

    # Tile inference (eval-only, can be turned off / on independently of training).
    # mode:
    #   'off'         — single-pass eval (cheapest)
    #   'pyramid_a'   — fixed 1+2x2 pyramid with 30% overlap (default; recommended)
    #   'hybrid_d'    — pyramid_a then re-tile the highest-scoring tile into 2x2
    # for_sources controls which dataset sources actually use tiling.  HOTS
    # scenes are small enough to skip tiling by default.
    # tile_cfg = dict(
    #     mode                = 'pyramid_a',
    #     levels              = (1, 2),
    #     overlap             = 0.30,
    #     for_sources         = ('insdet',),
    #     nms_iou             = 0.5,
    #     top_k               = 100,
    #     score_combo         = 'existence_x_score',
    #     edge_score_penalty  = 0.5,
    #     edge_px             = 4,
    #     merge_partial_boxes = True,
    #     merge_min_score     = 0.2,
    # ),
    tile_cfg = {'mode': 'off'}
)

---
## Phase 0 — Zero-shot OWLv2 baseline (no training)

**Goal**: Establish the floor. OWLv2 is pre-trained on COCO + Objects365, so it already knows everyday indoor objects. We run image-guided detection: each of the 4 supports independently → average per-patch logits → pick top-1 box.

Runs on:
- `vizwiz_novel` (16 instances, 1 image each, rotation-synthesised supports)
- `test` split of HOTS + InsDet

**Decision gate**: if mAP@50 < 5% on **both** datasets, abort and inspect annotations.

In [12]:
# Phase 0 — zero-shot OWLv2 baseline.
train_phase0(**SHARED_KWARGS, resume=False)

=== Phase 0 (zero-shot OWLv2) on cuda ===


Loading weights: 100%|██████████| 418/418 [00:00<00:00, 2752.23it/s]


phase0/vizwiz_novel: 16 instances
  phase0: 4 batches, tile_mode=off
  [1/4 =  25.0%]  elapsed=   5.9s  eta=  17.6s  rate=0.17 batch/s
  [2/4 =  50.0%]  elapsed=   9.6s  eta=   9.6s  rate=0.21 batch/s
  [3/4 =  75.0%]  elapsed=  13.4s  eta=   4.5s  rate=0.22 batch/s
  [4/4 = 100.0%]  elapsed=  17.1s  eta=   0.0s  rate=0.23 batch/s
vizwiz_novel done in 17.14s
phase0/test: 29 instances, 200 episodes
  phase0: 50 batches, tile_mode=off
  [1/50 =   2.0%]  elapsed=   4.6s  eta= 227.6s  rate=0.22 batch/s
  [2/50 =   4.0%]  elapsed=   8.4s  eta= 202.7s  rate=0.24 batch/s
  [3/50 =   6.0%]  elapsed=  12.2s  eta= 191.8s  rate=0.25 batch/s
  [4/50 =   8.0%]  elapsed=  16.0s  eta= 184.4s  rate=0.25 batch/s
  [5/50 =  10.0%]  elapsed=  19.8s  eta= 178.6s  rate=0.25 batch/s
  [6/50 =  12.0%]  elapsed=  23.6s  eta= 173.3s  rate=0.25 batch/s
  [7/50 =  14.0%]  elapsed=  27.5s  eta= 168.7s  rate=0.25 batch/s
  [8/50 =  16.0%]  elapsed=  31.3s  eta= 164.2s  rate=0.26 batch/s
  [9/50 =  18.0%]  elapsed=

{'vizwiz_novel': {'overall': {'n': 16,
   'n_pos': 16,
   'n_neg': 0,
   'iou_mean': 0.14070053095929325,
   'iou_median': 0.0897304005920887,
   'iou_p25': 0.04307750053703785,
   'iou_p75': 0.16984134539961815,
   'contain_mean': 0.6766362898051739,
   'contain_at_iou_50': 0.0625,
   'contain_at_iou_75': 0.0,
   'ap_per_iou': {'0.50': 0.06930693069306931,
    '0.55': 0.0,
    '0.60': 0.0,
    '0.65': 0.0,
    '0.70': 0.0,
    '0.75': 0.0,
    '0.80': 0.0,
    '0.85': 0.0,
    '0.90': 0.0,
    '0.95': 0.0},
   'map_50': 0.06930693069306931,
   'map_75': 0.0,
   'map_5095': 0.006930693069306932,
   'map_50_existence_only': 0.06930693069306931,
   'map_5095_existence_only': 0.006930693069306932,
   'map_50_score_only': 0.06930693069306931,
   'map_5095_score_only': 0.006930693069306932,
   'precision_50': 0.0625,
   'recall_50': 0.0625,
   'f1_50': 0.0625,
   'existence_acc': 1.0,
   'existence_acc_pos': 1.0,
   'existence_acc_neg': 0.0,
   'existence_auroc': 0.0,
   'existence_pr_auc':

### Evaluate Phase 0 on the test split

Re-runs the zero-shot baseline on the held-out test split only (HOTS + InsDet).

In [13]:
evaluate_phase0(**EVAL_KWARGS)

=== Phase 0 evaluation on test split (tile_mode=off) ===


Loading weights: 100%|██████████| 418/418 [00:00<00:00, 5920.32it/s]


  phase0: 50 batches, tile_mode=off
  [1/50 =   2.0%]  elapsed=   4.8s  eta= 233.0s  rate=0.21 batch/s
  [2/50 =   4.0%]  elapsed=   8.7s  eta= 208.4s  rate=0.23 batch/s
  [3/50 =   6.0%]  elapsed=  12.6s  eta= 197.7s  rate=0.24 batch/s
  [4/50 =   8.0%]  elapsed=  16.6s  eta= 190.4s  rate=0.24 batch/s
  [5/50 =  10.0%]  elapsed=  20.5s  eta= 184.5s  rate=0.24 batch/s
  [6/50 =  12.0%]  elapsed=  24.4s  eta= 179.2s  rate=0.25 batch/s
  [7/50 =  14.0%]  elapsed=  28.4s  eta= 174.3s  rate=0.25 batch/s
  [8/50 =  16.0%]  elapsed=  32.3s  eta= 169.7s  rate=0.25 batch/s
  [9/50 =  18.0%]  elapsed=  36.3s  eta= 165.2s  rate=0.25 batch/s
  [10/50 =  20.0%]  elapsed=  40.2s  eta= 160.8s  rate=0.25 batch/s
  [11/50 =  22.0%]  elapsed=  44.1s  eta= 156.5s  rate=0.25 batch/s
  [12/50 =  24.0%]  elapsed=  48.1s  eta= 152.3s  rate=0.25 batch/s
  [13/50 =  26.0%]  elapsed=  52.0s  eta= 148.1s  rate=0.25 batch/s
  [14/50 =  28.0%]  elapsed=  56.0s  eta= 144.0s  rate=0.25 batch/s
  [15/50 =  30.0%]  e

{'overall': {'n': 200,
  'n_pos': 89,
  'n_neg': 111,
  'iou_mean': 0.18793255168149312,
  'iou_median': 0.00045996648259460926,
  'iou_p25': 0.0,
  'iou_p75': 0.01717689447104931,
  'contain_mean': 0.5216670551996553,
  'contain_at_iou_50': 0.24719101123595505,
  'contain_at_iou_75': 0.14606741573033707,
  'ap_per_iou': {'0.50': 0.04899838821091411,
   '0.55': 0.04899838821091411,
   '0.60': 0.046913969242701525,
   '0.65': 0.046913969242701525,
   '0.70': 0.046913969242701525,
   '0.75': 0.02860865033871808,
   '0.80': 0.02018084161357312,
   '0.85': 0.00010589294223540001,
   '0.90': 0.0,
   '0.95': 0.0},
  'map_50': 0.04899838821091411,
  'map_75': 0.02860865033871808,
  'map_5095': 0.028763406904445937,
  'map_50_existence_only': 0.04899838821091411,
  'map_5095_existence_only': 0.028763406904445937,
  'map_50_score_only': 0.04899838821091411,
  'map_5095_score_only': 0.028763406904445937,
  'precision_50': 0.11518324607329843,
  'recall_50': 0.24719101123595505,
  'f1_50': 0.1571

---
## Stage 1.1 — Aggregator + Existence Head Warmup

**Goal**: Train the multi-view aggregator and existence head while keeping OWLv2 entirely frozen.

- **Trainable**: aggregator (~3M params), existence head (~5K params).
- **Frozen**: OWLv2 vision_model, class_head, box_head.
- **Loss**: focal existence loss + small existence-mean KL anti-collapse.
- **Per-stage layout**: `epochs × K=5 folds`, every (epoch, fold) writes `ckpt_foldF_epochE.pt` + `analysis/stage_1_1/epoch_E/fold_F.json`.

In [14]:
# Stage 1.1 — aggregator + existence head warmup.  With the four
# interventions in place (margin loss, prototype contrastive loss, richer
# existence features, higher existence LR) the head should escape the
# constant-output collapse within a few epochs, so we keep this stage
# short.
train_stage_1_1(
    **SHARED_KWARGS,
    stage_1_1_epochs = 4,
    resume = False,
)


=== Stage 1_1 on cuda ===


Loading weights: 100%|██████████| 418/418 [00:00<00:00, 5459.04it/s]


▶ stage 1_1 epoch 1/4 fold 0/4
  training : 50 batches


KeyboardInterrupt: 

### Evaluate Stage 1.1 on the test split

In [ ]:
# Tiling is on by default for InsDet (see EVAL_KWARGS['tile_cfg']).
# To disable for this run only: replace **EVAL_KWARGS with **{**EVAL_KWARGS, 'tile_cfg': {'mode': 'off'}}
evaluate_run(
    checkpoint=f'{OUT_ROOT}/stage_1_1/stage_complete.pt',
    **EVAL_KWARGS,
)

---
## Stage 1.2 — Detection Head Fine-Tuning

**Goal**: Adapt OWLv2's `class_head` and `box_head` to our domain and instance-matching task. This is where most of the mAP gain happens.

- **Trainable**: aggregator, existence head, OWLv2 class_head, OWLv2 box_head, OWLv2 layer_norm.
- **Frozen**: OWLv2 vision_model.
- **Loss**: focal + L1 + GIoU + anti-collapse.
- **Box-head warmup**: the box head is frozen for the first 3 epochs so the existence head can calibrate before bbox loss fires.

In [ ]:
# Stage 1.2 — class_head + box_head fine-tuning.  This is where most of
# the mAP gain on InsDet is expected to land (the class head's logit shift /
# scale finally adapt to our prototype distribution).  Box head warmup of
# 3 epochs lets the existence head settle before bbox loss starts firing.
train_stage_1_2(
    **SHARED_KWARGS,
    stage_1_2_epochs = 16,
)

### Evaluate Stage 1.2 on the test split

In [ ]:
evaluate_run(
    checkpoint=f'{OUT_ROOT}/stage_1_2/stage_complete.pt',
    **EVAL_KWARGS,
)

---
## Stage 2.3 — LoRA Fine-Tuning of last 4 ViT blocks

**Goal**: Inject low-rank adapters into the q/v projections of OWLv2's last 4 vision transformer blocks. The frozen ViT acts as a regulariser; the backbone adapts slightly without forgetting general visual structure.

- **Trainable**: aggregator, existence head, class_head, box_head, LoRA adapters (~98K params).
- **Frozen**: rest of OWLv2 vision_model.
- **Loss**: same as Stage 1.2.

In [ ]:
# Stage 2.3 — LoRA fine-tuning of the last 4 vision transformer blocks.
# LoRA activates gradients through the vision blocks, roughly doubling
# VRAM relative to Stage 1.2.  Use the measured safe config:
#   img=768, B=2 → 3.89 GB on RTX 2060 6 GB  (vs 5.60 GB at img=960, B=2)
# Switching from img=960 (stages 1.x) to img=768 here is fine: OWLv2
# uses interpolate_pos_encoding=True and the aggregator is patch-count-
# agnostic, so no weights are shape-tied to the resolution.
train_stage_2_3(
    **SHARED_KWARGS,
    stage_2_3_epochs = 8,
    img_size   = _S23_IMG_SIZE,
    batch_size = _S23_BATCH,
    grad_accum_steps = _S23_ACCUM,
)

### Evaluate Stage 2.3 on the test split

In [ ]:
evaluate_run(
    checkpoint=f'{OUT_ROOT}/stage_2_3/stage_complete.pt',
    **EVAL_KWARGS,
)

---
## Generate the offline report

Reads every per-(epoch, fold) JSON + aggregates + Phase 0 results, writes plot PNGs to `analysis/plots/`.

In [ ]:
plot_all_from_jsons(ANALYSIS_ROOT)

---
## Optional: export the trained model to ONNX / TFLite

Use the root-level `export.py` script.  ONNX is the primary target; TFLite is best-effort via `onnx2tf` (the OWLv2 attention stack does not have a clean direct TFLite path).  LoRA adapters are merged into the base weights before tracing.

In [ ]:
# Export the best Stage 2.3 checkpoint to ONNX.  Uncomment to run.
# !uv run python export.py \
#     --checkpoint checkpoints/stage_2_3/best.pt \
#     --out-dir exports/ \
#     --img-size {_DEFAULT_IMG_SIZE} \
#     --formats onnx

---
## Optional: ad-hoc inference on your own images

`inference.py::run_inference` takes 4 support paths + 1 query path and writes the originals + an annotated result image (with bbox + EXISTS/NOT EXISTS caption) to `inference/<NNNN>/`.  The tile mode is fully configurable per-call; pass `tile_cfg={'mode': 'off'}` to disable tiling, `'pyramid_a'` (default) for the recommended 1+2x2 pyramid, or `'hybrid_d'` for the recursive variant.

In [ ]:
# Example.  Replace the support / query paths with your own files.
# from inference import run_inference
#
# run_inference(
#     checkpoint=f'{OUT_ROOT}/stage_2_3/best.pt',
#     support_paths=['supports/s1.jpg', 'supports/s2.jpg', 'supports/s3.jpg', 'supports/s4.jpg'],
#     query_path='query/scene.jpg',
#     img_size=_DEFAULT_IMG_SIZE,
#     tile_cfg={'mode': 'pyramid_a'},   # or 'off' / 'hybrid_d'
# )